# CoherenceBench-IN — Phase 3: Evaluation Harness
**Weeks 11–18 | ~40–50 hours**

### Goals
Run the 584-instance English benchmark against multiple LLMs and collect accuracy by dimension, distance, and context length.

| Model | Method | Size | Expected runtime |
|-------|--------|------|------------------|
| `Llama-3.2-3B-Instruct` (4-bit) | mlx-lm (local M1) | 3B | ~1–2 hr |
| `Qwen2.5-7B-Instruct` (4-bit) | mlx-lm (local M1) | 7B | ~2–4 hr |
| `Mistral-7B-Instruct-v0.3` (4-bit) | mlx-lm (local M1) | 7B | ~2–4 hr |
| `gpt-4o-mini` | OpenAI API | — | ~15 min |
| `gpt-4o` | OpenAI API | — | ~30 min |

### Evaluation protocol
- **Prompt:** System instruction + full document text (truncated to `MAX_CTX_TOKENS`) + question
- **Output:** Parse first occurrence of `CONSISTENT` or `INCONSISTENT` (case-insensitive) from response
- **Metric:** Accuracy (correct / total) per model × dimension × distance × length bucket
- **Checkpointing:** Results saved incrementally — safe to interrupt and resume

### Phase 3 exit criteria
- [ ] ≥3 models evaluated on all 584 instances
- [ ] Results saved to `results/` as JSONL per model
- [ ] Main accuracy table generated (models × dimensions)
- [ ] Distance breakdown table generated
- [ ] Context-length analysis plot saved
- [ ] Notes added to `references/research_log.md`

In [1]:
# ─── Imports & Configuration ──────────────────────────────────────
import os, json, re, time
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from tqdm.auto import tqdm
from IPython.display import display, Markdown

# ── Paths ─────────────────────────────────────────────────────────
BASE         = Path("../")
DATA_BENCH   = BASE / "data" / "benchmark" / "english"
RESULTS_DIR  = BASE / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Evaluation settings ───────────────────────────────────────────
MAX_CTX_TOKENS  = 16_000   # truncate documents longer than this
MAX_NEW_TOKENS  = 15       # we only need one word; 15 is generous
RANDOM_SEED     = 42

# ── System prompt ─────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a precise document analyst. "
    "Read the document carefully and answer the question about its internal consistency. "
    "Your answer must be exactly ONE word: either CONSISTENT or INCONSISTENT. "
    "Do not explain. Do not add punctuation. Output only the single word."
)

print("✅ Imports ready.")
print(f"   Benchmark dir: {DATA_BENCH}")
print(f"   Results dir:   {RESULTS_DIR}")
print(f"   Max ctx:       {MAX_CTX_TOKENS:,} tokens")

✅ Imports ready.
   Benchmark dir: ../data/benchmark/english
   Results dir:   ../results
   Max ctx:       16,000 tokens


In [2]:
# ─── Load Benchmark ───────────────────────────────────────────────
INDEX_FILE = DATA_BENCH / "index.jsonl"

instances = []
with open(INDEX_FILE) as f:
    for line in f:
        instances.append(json.loads(line))

df_bench = pd.DataFrame(instances)
print(f"Loaded {len(instances)} instances")
print(df_bench.groupby(["dimension", "answer"]).size().unstack(fill_value=0))
print(f"\nContext length stats (tokens):")
print(df_bench["context_tokens"].describe().round(0))

# Build a lookup: instance id → full text path
def load_instance_text(inst_id: str) -> str:
    """Load the (possibly corrupted) text for an instance from its JSON file."""
    fpath = DATA_BENCH / f"{inst_id}.json"
    with open(fpath) as f:
        return json.load(f)["text"]

print("\n✅ Benchmark loaded.")

Loaded 584 instances
answer              CONSISTENT  INCONSISTENT
dimension                                   
causal_chain               100           100
entity_consistency          92            92
temporal_coherence         100           100

Context length stats (tokens):
count      584.0
mean      8617.0
std       4482.0
min       4014.0
25%       5214.0
50%       7439.0
75%      10219.0
max      28355.0
Name: context_tokens, dtype: float64

✅ Benchmark loaded.


In [3]:
# ─── Prompt Builder & Answer Parser ──────────────────────────────
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

def build_prompt(text: str, question: str, max_ctx_tokens: int = MAX_CTX_TOKENS) -> str:
    """
    Build the user-turn prompt. Truncates the document if it would exceed
    max_ctx_tokens total (leaving ~512 tokens for the question and formatting).
    """
    q_tokens  = len(enc.encode(question))
    overhead  = 512   # system prompt + formatting
    doc_budget = max_ctx_tokens - q_tokens - overhead

    doc_tokens = enc.encode(text)
    if len(doc_tokens) > doc_budget:
        text = enc.decode(doc_tokens[:doc_budget]) + " [...document truncated...]"

    return (
        f"Document:\n{text}\n\n"
        f"---\n{question}"
    )


def parse_answer(response: str) -> str:
    """
    Extract CONSISTENT or INCONSISTENT from model response.
    Returns 'INCONSISTENT', 'CONSISTENT', or 'UNPARSEABLE'.
    """
    # Check for INCONSISTENT first (it contains CONSISTENT as substring)
    if re.search(r"\bINCONSISTENT\b", response, re.IGNORECASE):
        return "INCONSISTENT"
    if re.search(r"\bCONSISTENT\b", response, re.IGNORECASE):
        return "CONSISTENT"
    return "UNPARSEABLE"


def ctx_length_bucket(n_tokens: int) -> str:
    """Bucket context length for analysis."""
    if n_tokens < 6_000:  return "4K–6K"
    if n_tokens < 12_000: return "6K–12K"
    return "12K+"


# Quick sanity check on parser
assert parse_answer("INCONSISTENT")              == "INCONSISTENT"
assert parse_answer("The answer is CONSISTENT.") == "CONSISTENT"
assert parse_answer("inconsistent")              == "INCONSISTENT"
assert parse_answer("I don't know")              == "UNPARSEABLE"
print("✅ Prompt builder and answer parser ready.")

✅ Prompt builder and answer parser ready.


---
## Part 1 — Local Models (mlx-lm, Apple Silicon)

Uses 4-bit quantized models from `mlx-community` on HuggingFace.
Downloads happen on first run (~2–5 GB per model, cached in `~/.cache/huggingface/`).

Results are saved incrementally — **safe to interrupt and re-run** (already-evaluated instances are skipped).

**Run one model at a time.** After each model completes, scroll to Part 3 to check preliminary results before running the next.

In [4]:
# ─── MLX-LM Inference Function ────────────────────────────────────
def run_mlx_model(hf_model_id: str, run_name: str,
                  instances: list = instances) -> pd.DataFrame:
    """
    Run all benchmark instances through a local mlx-lm model.
    Saves results to results/{run_name}_results.jsonl (checkpointed, resumable).

    Parameters
    ----------
    hf_model_id : str   e.g. 'mlx-community/Llama-3.2-3B-Instruct-4bit'
    run_name    : str   short name used for the output file, e.g. 'llama3_3b'
    """
    from mlx_lm import load, generate as mlx_generate

    results_file = RESULTS_DIR / f"{run_name}_results.jsonl"

    # Load already-done IDs (for resumption after interruption)
    done_ids: set = set()
    if results_file.exists():
        with open(results_file) as f:
            for line in f:
                done_ids.add(json.loads(line)["id"])
        print(f"  Resuming — {len(done_ids)} instances already done.")

    remaining = [inst for inst in instances if inst["id"] not in done_ids]
    if not remaining:
        print(f"  {run_name}: all {len(instances)} instances already evaluated.")
    else:
        print(f"  Loading model: {hf_model_id} ...")
        model, tokenizer = load(hf_model_id)
        print(f"  Model loaded. Evaluating {len(remaining)} instances...")

        unparseable = 0
        with open(results_file, "a") as out_f:
            for inst in tqdm(remaining, desc=run_name):
                text     = load_instance_text(inst["id"])
                user_msg = build_prompt(text, inst["question"])

                # Format with chat template
                messages  = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ]
                formatted = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )

                response = mlx_generate(
                    model, tokenizer,
                    prompt    = formatted,
                    max_tokens= MAX_NEW_TOKENS,
                    verbose   = False,
                )

                pred    = parse_answer(response)
                correct = (pred == inst["answer"])
                if pred == "UNPARSEABLE":
                    unparseable += 1

                record = {
                    "id":             inst["id"],
                    "dimension":      inst["dimension"],
                    "distance":       inst["distance"],
                    "context_tokens": inst["context_tokens"],
                    "gold":           inst["answer"],
                    "pred":           pred,
                    "correct":        correct,
                    "raw_response":   response.strip()[:120],
                }
                out_f.write(json.dumps(record) + "\n")
                out_f.flush()

        del model  # free unified memory before loading next model
        print(f"  ✅ {run_name} done. Unparseable: {unparseable}/{len(remaining)}")

    # Load and return full results as DataFrame
    rows = []
    with open(results_file) as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


print("✅ run_mlx_model() defined.")
print("   → Next: run the model cells below one at a time.")

✅ run_mlx_model() defined.
   → Next: run the model cells below one at a time.


In [ ]:
# ─── Model 1: Llama-3.2-3B-Instruct (4-bit) ─────────────────────
# Download: ~1.8 GB. Runtime: ~1–2 hr for 584 instances on M1 16GB.
df_llama3 = run_mlx_model(
    hf_model_id = "mlx-community/Llama-3.2-3B-Instruct-4bit",
    run_name    = "llama3_3b",
)
acc = df_llama3["correct"].mean()
print(f"\nLlama-3.2-3B  overall accuracy: {acc:.1%}")
print(df_llama3.groupby("dimension")["correct"].mean().round(3).to_string())

In [ ]:
# ─── Model 2: Qwen2.5-7B-Instruct (4-bit) ───────────────────────
# Download: ~4.5 GB. Runtime: ~2–4 hr for 584 instances on M1 16GB.
df_qwen = run_mlx_model(
    hf_model_id = "mlx-community/Qwen2.5-7B-Instruct-4bit",
    run_name    = "qwen25_7b",
)
acc = df_qwen["correct"].mean()
print(f"\nQwen2.5-7B  overall accuracy: {acc:.1%}")
print(df_qwen.groupby("dimension")["correct"].mean().round(3).to_string())

In [ ]:
# ─── Model 3: Mistral-7B-Instruct-v0.3 (4-bit) ───────────────────
# Download: ~4.5 GB. Runtime: ~2–4 hr for 584 instances on M1 16GB.
df_mistral = run_mlx_model(
    hf_model_id = "mlx-community/Mistral-7B-Instruct-v0.3-4bit",
    run_name    = "mistral_7b",
)
acc = df_mistral["correct"].mean()
print(f"\nMistral-7B  overall accuracy: {acc:.1%}")
print(df_mistral.groupby("dimension")["correct"].mean().round(3).to_string())

---
## Part 2 — API Models (OpenAI)

Set your API key before running:
```python
import os
os.environ["OPENAI_API_KEY"] = "sk-..."
```
Or set it in your shell: `export OPENAI_API_KEY=sk-...`

**Cost estimate:**
- `gpt-4o-mini`: 584 × ~10K input tokens ≈ 5.84M tokens → ~**$0.87**
- `gpt-4o`:      584 × ~10K input tokens ≈ 5.84M tokens → ~**$14.60**

Rate limiting: 1 request/sec to stay within tier-1 limits.

In [5]:
# ─── OpenAI Inference Function ────────────────────────────────────
from openai import OpenAI

def run_openai_model(model_id: str, run_name: str,
                     instances: list = instances,
                     requests_per_sec: float = 1.0) -> pd.DataFrame:
    """
    Run all benchmark instances through an OpenAI model.
    Checkpointed — safe to interrupt and resume.
    """
    api_key = os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        raise ValueError(
            "OPENAI_API_KEY not set.\n"
            "  Run: import os; os.environ['OPENAI_API_KEY'] = 'sk-...'"
        )

    client       = OpenAI(api_key=api_key)
    results_file = RESULTS_DIR / f"{run_name}_results.jsonl"
    min_interval = 1.0 / requests_per_sec

    done_ids: set = set()
    if results_file.exists():
        with open(results_file) as f:
            for line in f:
                done_ids.add(json.loads(line)["id"])
        print(f"  Resuming — {len(done_ids)} instances already done.")

    remaining = [inst for inst in instances if inst["id"] not in done_ids]
    if not remaining:
        print(f"  {run_name}: all {len(instances)} instances already evaluated.")
    else:
        print(f"  Evaluating {len(remaining)} instances with {model_id}...")
        unparseable = 0
        last_call   = 0.0

        with open(results_file, "a") as out_f:
            for inst in tqdm(remaining, desc=run_name):
                # Rate limiting
                elapsed = time.time() - last_call
                if elapsed < min_interval:
                    time.sleep(min_interval - elapsed)

                text     = load_instance_text(inst["id"])
                user_msg = build_prompt(text, inst["question"])

                # Retry on transient errors
                for attempt in range(4):
                    try:
                        resp = client.chat.completions.create(
                            model    = model_id,
                            messages = [
                                {"role": "system", "content": SYSTEM_PROMPT},
                                {"role": "user",   "content": user_msg},
                            ],
                            max_tokens  = MAX_NEW_TOKENS,
                            temperature = 0.0,
                        )
                        response = resp.choices[0].message.content or ""
                        last_call = time.time()
                        break
                    except Exception as e:
                        wait = 2 ** attempt
                        print(f"    API error ({e}), retrying in {wait}s...")
                        time.sleep(wait)
                        response = ""

                pred    = parse_answer(response)
                correct = (pred == inst["answer"])
                if pred == "UNPARSEABLE":
                    unparseable += 1

                record = {
                    "id":             inst["id"],
                    "dimension":      inst["dimension"],
                    "distance":       inst["distance"],
                    "context_tokens": inst["context_tokens"],
                    "gold":           inst["answer"],
                    "pred":           pred,
                    "correct":        correct,
                    "raw_response":   response.strip()[:120],
                }
                out_f.write(json.dumps(record) + "\n")
                out_f.flush()

        print(f"  ✅ {run_name} done. Unparseable: {unparseable}/{len(remaining)}")

    rows = []
    with open(results_file) as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


print("✅ run_openai_model() defined.")

✅ run_openai_model() defined.


In [ ]:
# ─── Model 4: gpt-4o-mini ─────────────────────────────────────────
# Requires: os.environ["OPENAI_API_KEY"] to be set.
# Cost: ~$0.87 for full 584 instances.

# import os; os.environ["OPENAI_API_KEY"] = "sk-..."  # ← uncomment & fill in

df_gpt4o_mini = run_openai_model(
    model_id  = "gpt-4o-mini",
    run_name  = "gpt4o_mini",
)
acc = df_gpt4o_mini["correct"].mean()
print(f"\ngpt-4o-mini  overall accuracy: {acc:.1%}")
print(df_gpt4o_mini.groupby("dimension")["correct"].mean().round(3).to_string())

In [ ]:
# ─── Model 5: gpt-4o ──────────────────────────────────────────────
# Cost: ~$14.60 for full 584 instances.
# Run this last — highest cost.

# import os; os.environ["OPENAI_API_KEY"] = "sk-..."  # ← uncomment & fill in

df_gpt4o = run_openai_model(
    model_id  = "gpt-4o",
    run_name  = "gpt4o",
)
acc = df_gpt4o["correct"].mean()
print(f"\ngpt-4o  overall accuracy: {acc:.1%}")
print(df_gpt4o.groupby("dimension")["correct"].mean().round(3).to_string())

---
## Part 3 — Results Analysis & Paper Tables
Run these cells once all desired models are evaluated.
Produces Table 3 (main results), Table 4 (distance breakdown), and Figure 2 (context-length curve) from the paper.

In [6]:
# ─── Load All Available Results ───────────────────────────────────
MODEL_DISPLAY_NAMES = {
    "llama3_3b":  "Llama-3.2-3B",
    "qwen25_7b":  "Qwen2.5-7B",
    "mistral_7b": "Mistral-7B",
    "gpt4o_mini": "GPT-4o-mini",
    "gpt4o":      "GPT-4o",
}

all_results: dict[str, pd.DataFrame] = {}
for run_name, display_name in MODEL_DISPLAY_NAMES.items():
    fpath = RESULTS_DIR / f"{run_name}_results.jsonl"
    if fpath.exists():
        rows = [json.loads(l) for l in open(fpath)]
        df   = pd.DataFrame(rows)
        # Add context length bucket
        df["ctx_bucket"] = df["context_tokens"].apply(ctx_length_bucket)
        all_results[display_name] = df
        n_parseable = (df["pred"] != "UNPARSEABLE").sum()
        print(f"  {display_name:<18}: {len(df):>4} instances, "
              f"{n_parseable} parseable ({n_parseable/len(df)*100:.1f}%)")
    else:
        print(f"  {display_name:<18}: not yet evaluated")

if not all_results:
    print("\n⚠️  No model results found yet. Run Part 1 or Part 2 cells first.")
else:
    print(f"\nLoaded {len(all_results)} model(s): {', '.join(all_results.keys())}")

  Llama-3.2-3B      : not yet evaluated
  Qwen2.5-7B        : not yet evaluated
  Mistral-7B        : not yet evaluated
  GPT-4o-mini       : not yet evaluated
  GPT-4o            : not yet evaluated

⚠️  No model results found yet. Run Part 1 or Part 2 cells first.


In [ ]:
# ─── Table 3: Main Results (accuracy by model × dimension) ────────
if not all_results:
    print("No results yet.")
else:
    DIMS = ["entity_consistency", "temporal_coherence", "causal_chain"]
    DIM_SHORT = {"entity_consistency": "Entity",
                 "temporal_coherence": "Temporal",
                 "causal_chain":       "Causal"}

    rows_table = []
    for model_name, df in all_results.items():
        row = {"Model": model_name}
        overall_correct = df["correct"].sum()
        overall_total   = len(df)
        for dim in DIMS:
            sub = df[df["dimension"] == dim]
            acc = sub["correct"].mean() if len(sub) > 0 else float("nan")
            row[DIM_SHORT[dim]] = f"{acc:.1%}"
        row["Overall"] = f"{overall_correct/overall_total:.1%}"
        rows_table.append(row)

    df_table3 = pd.DataFrame(rows_table).set_index("Model")

    print("\nTable 3 — Accuracy by Dimension")
    print("=" * 60)
    display(df_table3)

    # Save as CSV
    df_table3.to_csv(RESULTS_DIR / "table3_accuracy_by_dimension.csv")
    print(f"\nSaved: {RESULTS_DIR / 'table3_accuracy_by_dimension.csv'}")

In [ ]:
# ─── Table 4: Distance Breakdown (accuracy by model × distance) ───
if not all_results:
    print("No results yet.")
else:
    rows_dist = []
    for model_name, df in all_results.items():
        # Only incoherent instances have a meaningful distance
        inc_df = df[df["gold"] == "INCONSISTENT"]
        row = {"Model": model_name}
        for dist in ["near", "mid", "far"]:
            sub = inc_df[inc_df["distance"] == dist]
            acc = sub["correct"].mean() if len(sub) > 0 else float("nan")
            row[dist.capitalize()] = f"{acc:.1%}"
        rows_dist.append(row)

    df_table4 = pd.DataFrame(rows_dist).set_index("Model")
    print("\nTable 4 — Accuracy on INCONSISTENT instances by Corruption Distance")
    print("=" * 60)
    display(df_table4)

    df_table4.to_csv(RESULTS_DIR / "table4_accuracy_by_distance.csv")
    print(f"\nSaved: {RESULTS_DIR / 'table4_accuracy_by_distance.csv'}")

In [ ]:
# ─── Figure 2: Accuracy vs Context Length ─────────────────────────
# Shows how models degrade as document length increases.
if not all_results:
    print("No results yet.")
else:
    BUCKET_ORDER = ["4K–6K", "6K–12K", "12K+"]
    colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, dim in zip(axes, ["entity_consistency", "temporal_coherence", "causal_chain"]):
        for (model_name, df), color in zip(all_results.items(), colors):
            sub = df[df["dimension"] == dim]
            accs = [
                sub[sub["ctx_bucket"] == b]["correct"].mean()
                for b in BUCKET_ORDER
            ]
            ax.plot(BUCKET_ORDER, accs, marker="o", label=model_name,
                    color=color, linewidth=2, markersize=6)

        ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.6, label="Chance (50%)")
        ax.set_title(DIM_SHORT.get(dim, dim), fontsize=12, fontweight="bold")
        ax.set_xlabel("Context length (tokens)")
        ax.set_ylim(0, 1.05)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
        ax.grid(axis="y", alpha=0.3)

    axes[0].set_ylabel("Accuracy")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(all_results)+1,
               bbox_to_anchor=(0.5, -0.12), fontsize=9)
    plt.suptitle("Figure 2 — Accuracy vs. Context Length by Dimension",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "fig2_accuracy_vs_context_length.png",
                dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {RESULTS_DIR / 'fig2_accuracy_vs_context_length.png'}")

In [ ]:
# ─── Figure 3: Main Results Bar Chart ────────────────────────────
if not all_results:
    print("No results yet.")
else:
    DIM_COLS = ["entity_consistency", "temporal_coherence", "causal_chain"]
    model_names = list(all_results.keys())
    x = np.arange(len(DIM_COLS))
    width = 0.8 / max(len(model_names), 1)

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (model_name, df) in enumerate(all_results.items()):
        accs = [df[df["dimension"] == d]["correct"].mean() for d in DIM_COLS]
        offset = (i - len(model_names)/2 + 0.5) * width
        bars = ax.bar(x + offset, accs, width * 0.9,
                      label=model_name, color=colors[i % len(colors)])
        for bar, acc in zip(bars, accs):
            if not np.isnan(acc):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f"{acc:.0%}", ha="center", va="bottom", fontsize=8)

    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1.5,
               alpha=0.7, label="Chance (50%)")
    ax.set_xticks(x)
    ax.set_xticklabels([DIM_SHORT[d] for d in DIM_COLS], fontsize=11)
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1.1)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_title("Figure 3 — Accuracy by Dimension", fontsize=12, fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "fig3_accuracy_by_dimension.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {RESULTS_DIR / 'fig3_accuracy_by_dimension.png'}")

In [ ]:
# ─── Error Analysis: Unparseable & False Rates ────────────────────
if not all_results:
    print("No results yet.")
else:
    print("Error Analysis")
    print("=" * 60)
    for model_name, df in all_results.items():
        total       = len(df)
        unparseable = (df["pred"] == "UNPARSEABLE").sum()
        fp          = ((df["gold"] == "CONSISTENT") & (df["pred"] == "INCONSISTENT")).sum()
        fn          = ((df["gold"] == "INCONSISTENT") & (df["pred"] == "CONSISTENT")).sum()
        print(f"\n{model_name}:")
        print(f"  Unparseable:     {unparseable:>4} / {total} ({unparseable/total:.1%})")
        print(f"  False Positive:  {fp:>4} (predicted INCONSISTENT when CONSISTENT)")
        print(f"  False Negative:  {fn:>4} (predicted CONSISTENT when INCONSISTENT)")

        # Show 3 example unparseable responses
        bad_examples = df[df["pred"] == "UNPARSEABLE"].head(3)
        if len(bad_examples) > 0:
            print(f"  Unparseable examples:")
            for _, row in bad_examples.iterrows():
                print(f"    [{row['id']}] raw: '{row['raw_response'][:80]}'")

In [ ]:
# ─── Phase 3 Summary ──────────────────────────────────────────────
evaluated = [m for m in MODEL_DISPLAY_NAMES.values()
             if (RESULTS_DIR / f"{[k for k,v in MODEL_DISPLAY_NAMES.items() if v==m][0]}_results.jsonl").exists()]

print("PHASE 3 SUMMARY")
print("=" * 50)
print(f"Models evaluated: {len(evaluated)} / 5")
for m in evaluated:
    print(f"  ✅ {m}")
remaining = [m for m in MODEL_DISPLAY_NAMES.values() if m not in evaluated]
for m in remaining:
    print(f"  ⬜ {m}")

print()
tables = list((RESULTS_DIR).glob("table*.csv"))
figs   = list((RESULTS_DIR).glob("fig*.png"))
print(f"[{'✅' if len(tables) >= 2 else '⬜'}] Result tables saved: {[t.name for t in tables]}")
print(f"[{'✅' if len(figs) >= 2 else '⬜'}] Figures saved:       {[f.name for f in figs]}")
print()
print("→ Next notebook: 04_paper_writing.ipynb  (Phase 4)")

---
## Phase 3 Checklist

| # | Task | Status | Notes |
|---|------|--------|-------|
| 1 | Install mlx-lm + resolve transformers version | ✅ | transformers 5.2.0, mlx-lm 0.30.7 |
| 2 | Define prompt builder + answer parser | ✅ | Handles INCONSISTENT⊃CONSISTENT substring issue |
| 3 | Define mlx-lm inference function (checkpointed) | ✅ | Resumable, saves per-instance |
| 4 | Define OpenAI inference function (checkpointed) | ✅ | Rate-limited, exponential backoff |
| 5 | Run Llama-3.2-3B | ⬜ | ~1–2 hr on M1 16GB |
| 6 | Run Qwen2.5-7B | ⬜ | ~2–4 hr on M1 16GB |
| 7 | Run Mistral-7B | ⬜ | ~2–4 hr on M1 16GB |
| 8 | Run gpt-4o-mini | ⬜ | ~$0.87, ~15 min |
| 9 | Run gpt-4o | ⬜ | ~$14.60, ~30 min |
| 10 | Generate Table 3 (accuracy by dimension) | ⬜ | Paper Table 3 |
| 11 | Generate Table 4 (accuracy by distance) | ⬜ | Paper Table 4 |
| 12 | Generate Figure 2 (accuracy vs context length) | ⬜ | Paper Figure 2 |
| 13 | Generate Figure 3 (bar chart by dimension) | ⬜ | Paper Figure 3 |
| 14 | Log findings in `references/research_log.md` | ⬜ | Key observations for paper §5 |

**Minimum for paper submission:** Tasks 5–8 + 10–13 (3 open-source + 1 API model, all tables/figures).